# Cosmic Ray Storm Prediction — Modelling

This notebook contains all modelling experiments. It depends on artifacts
produced by `cosmic_ray_storm_prediction.ipynb` (preprocessing, feature
engineering, feature selection).

**Depends on:**
- `data/processed/feat_split.parquet`
- `models/split_masks.pkl`
- `models/context_constants.pkl`
- `models/feature_selection_results.pkl`

## Experimental Design

| Stage | Description |
|---|---|
| 1 | Naive Persistence baseline |
| 2 | Direct AR baseline: $D_{st}(t+h) = \\alpha_h D_{st}(t) + \\beta_h$ |
| 3 | XGBoost — OMNI only (MODEL\_A) |
| 4 | XGBoost — OMNI + $\\delta n$ (MODEL\_C) |
| 5 | $\\Delta R^2$ analysis — H\_gain hypothesis |
| 6 | Horizon selection $h^*$ |
| 7 | Tuning on Train\_1 + Train\_2 |
| 8 | SHAP + feature importance |
| 9 | Storm analysis |
| 10 | Residual analysis |
| 11 | Final evaluation — Test\_Active + Test\_Quiet (once only) |

In [1]:
# ── Cell 1: Setup & Load ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')

import os
import json
import tempfile
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.estimators import NaivePersistence, DirectARBaseline, XGBoostDst
from src.evaluate import compute_metrics
from src.mlflow_tracking import setup_mlflow

In [2]:
setup_mlflow()

# ── Load artifacts ────────────────────────────────────────────────────────
feat  = pd.read_parquet('data/processed/feat_split.parquet')
masks = joblib.load('models/split_masks.pkl')
ctx   = joblib.load('models/context_constants.pkl')
fs    = joblib.load('models/feature_selection_results.pkl')

FEATURE_COLS      = ctx['FEATURE_COLS']
K_HORIZONS        = ctx['K_HORIZONS']
STORM_THR         = ctx['STORM_THR']
SELECTED_FEATURES = fs['selected_strict']
scaler = fs['scaler']

print(f'SELECTED_FEATURES : {len(SELECTED_FEATURES)}')
print(SELECTED_FEATURES)

print(f'feat shape        : {feat.shape}')
print(f'K_HORIZONS        : {K_HORIZONS}')
print(f'STORM_THR         : {STORM_THR} nT')
print(f'SELECTED_FEATURES : {len(SELECTED_FEATURES)}')

SELECTED_FEATURES : 23
['bz_gsm', 'sw_speed', 'sw_density', 'sw_pressure', 'e_field', 'mach_alfven', 'f107', 'ssn', 'neutron_counts', 'bz_acc_3h', 'bz_acc_6h', 'bz_acc_12h', 'bz_gsm_lag1', 'bz_gsm_lag3', 'bz_gsm_lag12', 'bz_gsm_lag21', 'sw_speed_lag1', 'sw_speed_lag3', 'sw_speed_lag7', 'neutron_counts_lag3', 'neutron_counts_lag7', 'solar_sin', 'solar_cos']
feat shape        : (364728, 73)
K_HORIZONS        : [1, 3, 7, 12, 21]
STORM_THR         : -50 nT
SELECTED_FEATURES : 23


## Experimental Design

Feature sets are frozen here before any modelling. This cell is the single
source of truth for which features enter each model. The design is motivated
by the two project hypotheses:

- **H\_skill:** XGBoost on OMNI achieves $R^2 \\geq 0.60$ at $h=7$h on the held-out test set.
- **H\_gain:** OMNI + $\\delta n(t)$ achieves higher $R^2$ at $h \\geq 7$h than OMNI alone.

The primary comparison is MODEL\_A vs MODEL\_C. Models B and D are ablation
variants that decompose the neutron contribution.

| Model | Features | Purpose |
|---|---|---|
| MODEL\_A | OMNI only | H\_skill baseline; H\_gain reference |
| MODEL\_C | OMNI + $\\delta n$ | Primary H\_gain test |
| MODEL\_B | OMNI + raw counts | Ablation: raw vs derived |
| MODEL\_D | OMNI + $\\delta n$ + lags | Ablation: lag contribution |

In [3]:
# ── Cell 2: Experimental Design — feature sets ────────────────────────────
#
# Feature sets are frozen here. Do not modify after first run.
# All subsequent modelling cells reference these constants.

NEUTRON_ALL = [
    'neutron_counts',
    'd_neutron',
    'neutron_counts_lag3',
    'neutron_counts_lag7',
]

NEUTRON_RAW     = ['neutron_counts']
NEUTRON_DERIVED = ['d_neutron']
NEUTRON_HISTORY = ['neutron_counts_lag3', 'neutron_counts_lag7']

OMNI_FEATURES = [
    f for f in SELECTED_FEATURES
    if f not in NEUTRON_ALL
]

# ── Main hypothesis models ────────────────────────────────────────────────
MODEL_A_OMNI          = OMNI_FEATURES
MODEL_C_OMNI_DNEUTRON = OMNI_FEATURES + NEUTRON_DERIVED

# ── Ablation models ───────────────────────────────────────────────────────
MODEL_B_OMNI_RAW     = OMNI_FEATURES + NEUTRON_RAW
MODEL_D_FULL_NEUTRON = OMNI_FEATURES + NEUTRON_DERIVED + NEUTRON_HISTORY

FEATURE_SETS = {
    'MODEL_A_OMNI'         : MODEL_A_OMNI,
    'MODEL_B_OMNI_RAW'     : MODEL_B_OMNI_RAW,
    'MODEL_C_OMNI_DNEUTRON': MODEL_C_OMNI_DNEUTRON,
    'MODEL_D_FULL_NEUTRON' : MODEL_D_FULL_NEUTRON,
}

EXPERIMENT_DESIGN = {
    'primary_comparison': {
        'baseline'     : 'MODEL_A_OMNI',
        'neutron_model': 'MODEL_C_OMNI_DNEUTRON',
        'hypothesis'   : 'H_gain',
    },
    'ablation': {
        'raw_neutron'    : ['MODEL_A_OMNI',          'MODEL_B_OMNI_RAW'],
        'neutron_history': ['MODEL_C_OMNI_DNEUTRON', 'MODEL_D_FULL_NEUTRON'],
    },
}

# ── Save to context_constants.pkl ─────────────────────────────────────────
ctx['OMNI_FEATURES']      = OMNI_FEATURES
ctx['NEUTRON_RAW']        = NEUTRON_RAW
ctx['NEUTRON_DERIVED']    = NEUTRON_DERIVED
ctx['NEUTRON_HISTORY']    = NEUTRON_HISTORY
ctx['FEATURE_SETS']       = FEATURE_SETS
ctx['EXPERIMENT_DESIGN']  = EXPERIMENT_DESIGN
joblib.dump(ctx, 'models/context_constants.pkl')

print(f'OMNI_FEATURES      : {len(OMNI_FEATURES)}')
print(f'MODEL_A (OMNI)     : {len(MODEL_A_OMNI)}')
print(f'MODEL_C (OMNI+δn)  : {len(MODEL_C_OMNI_DNEUTRON)}')
print(f'MODEL_B (OMNI+raw) : {len(MODEL_B_OMNI_RAW)}')
print(f'MODEL_D (full)     : {len(MODEL_D_FULL_NEUTRON)}')
print('context_constants.pkl updated')

OMNI_FEATURES      : 20
MODEL_A (OMNI)     : 20
MODEL_C (OMNI+δn)  : 21
MODEL_B (OMNI+raw) : 21
MODEL_D (full)     : 23
context_constants.pkl updated


In [4]:
# ── Cell 3: Train / Validation splits ────────────────────────────────────
#
# Train_1 is reconstructed from BOUNDARIES for Stages 3-5 (horizon selection
# and H_gain test). Train_2 is reserved for Stage 6 (tuning).
# masks['train'] = Train_1 | Train_2 — used in Stage 6.
#
# y_train is derived from Train_1 only — used as MASE denominator.

BOUNDARIES = ctx['BOUNDARIES']
PURGE_H    = ctx['PURGE_H']

dt = feat['datetime']

def segment_mask(start_key, end_key, purge_start=True, purge_end=True):
    """Boolean mask for a segment with optional purge zones."""
    start = BOUNDARIES[start_key]
    end   = BOUNDARIES[end_key]
    if purge_start:
        start = start + pd.Timedelta(hours=PURGE_H)
    if purge_end:
        end   = end   - pd.Timedelta(hours=PURGE_H)
    return (dt >= start) & (dt <= end)

train1_mask    = segment_mask('train1_start', 'train1_end',
                               purge_start=False, purge_end=True)
train2_mask    = segment_mask('train2_start', 'train2_end',
                               purge_start=True,  purge_end=True)
train_mask     = masks['train']       # Train_1 | Train_2 — for Stage 6
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

y_train = feat.loc[train1_mask, 'dst'].copy()

print(f'Train_1 rows    : {train1_mask.sum():,}')
print(f'Train_2 rows    : {train2_mask.sum():,}')
print(f'Train_1+2 rows  : {train_mask.sum():,}')
print(f'Val_main rows   : {val_main_mask.sum():,}')
print(f'Val_storm rows  : {val_storm_mask.sum():,}')
print(f'y_train NaN     : {y_train.isna().sum()}')

Train_1 rows    : 76,995
Train_2 rows    : 44,190
Train_1+2 rows  : 121,185
Val_main rows   : 52,542
Val_storm rows  : 1,446
y_train NaN     : 0


## Stage 1 — Naive Persistence

$$\hat{D}_{st}(t+h) = D_{st}(t)$$

The naive persistence forecast predicts the geomagnetic field $h$ hours ahead as identical to its current value. No learning occurs — `fit()` is a no-op. This is the zero-complexity baseline that defines the absolute lower bound for all subsequent models.

**Why persistence matters:** Any model with MASE > 1 at a given horizon provides no predictive value beyond the current observation, regardless of its absolute RMSE. Persistence performance degrades monotonically with horizon as the autocorrelation of $D_{st}$ decays from PACF lag1 = 0.978 toward zero.

**How it works:** For each forecast horizon $h \in \{1, 3, 7, 12, 21\}$, the current $D_{st}(t)$ is used directly as the prediction for $D_{st}(t+h)$. The forecast error $e_t^{(h)} = D_{st}(t+h) - D_{st}(t)$ is simply the $h$-step difference of the $D_{st}$ series. No features, no parameters, no training data are required.

**Pipeline:**
- `evaluate_persistence(seg_mask, h)` — extracts `X_seg` and `y_true` from `feat`, calls `NaivePersistence.predict()` which returns `X['dst']` directly, then passes predictions to `compute_metrics()` which returns RMSE, Storm RMSE, MASE, DM statistic and peak timing error.
- `log_metrics_run(run_name, tags, params, metrics, nested)` — opens a single MLflow run, sets tags, logs params and filters out NaN/inf values before logging metrics.
- `print_horizon_metrics(h, metrics)` — prints RMSE, Storm RMSE and MASE for one horizon as a progress indicator.
- `run_persistence_horizon(seg_name, seg_mask, h)` — orchestrates one horizon: calls `evaluate_persistence()`, delegates logging to `log_metrics_run()` as a nested child run, and calls `print_horizon_metrics()`.
- `run_persistence_segment(seg_name, seg_mask)` — opens a parent MLflow run for one segment and delegates to `run_persistence_horizon()` for each horizon in `K_HORIZONS`.

**Evaluation design:** The model is evaluated on two validation segments separately — `val_main` (2009–2014) for routine multi-horizon evaluation and `val_storm` (Halloween 2003, $D_{st}$ = −422 nT) for extreme event characterisation. Purge zones of 21h on each segment boundary prevent lag feature leakage across splits [LAP20]. The DM statistic equals 0 and p-value equals 1.0 by construction since persistence is compared against itself — this serves as a wiring sanity check for `compute_metrics()`.

**Reference thresholds:** Storm RMSE at h=7h on both validation segments establishes the primary operational target that all subsequent models must surpass. MASE = 1.0 is the boundary below which a model outperforms persistence at a given horizon.

In [5]:
# ── Cell 4: Naive Persistence ─────────────────────────────────────────────
#
# Helper functions are defined here and reused in Cells 5, 6, 7.
# Each function has a single responsibility.

def evaluate_persistence(seg_mask, h):
    """
    Compute persistence metrics for one segment at horizon h.
    Returns metrics dict from compute_metrics().
    """
    X_seg  = feat.loc[seg_mask].copy()
    y_true = feat.loc[seg_mask, f'dst_target_{h}h'].copy()
    y_pred = persistence_model.predict(X_seg)
    return compute_metrics(
        y_true    = y_true,
        y_pred    = y_pred,
        y_train   = y_train,
        storm_thr = STORM_THR,
        horizon   = h,
    )

def log_metrics_run(run_name, tags, params, metrics, nested=False):
    """
    Log a single MLflow run with tags, params and metrics.
    NaN and inf values are excluded from metrics logging.
    """
    with mlflow.start_run(run_name=run_name, nested=nested):
        for k, v in tags.items():
            mlflow.set_tag(k, v)
        mlflow.log_params(params)
        mlflow.log_metrics({
            k: float(v) for k, v in metrics.items()
            if isinstance(v, (int, float)) and np.isfinite(float(v))
        })

def print_horizon_metrics(h, metrics):
    """
    Print RMSE, Storm RMSE and MASE for one horizon.
    Used as progress indicator during evaluation loops.
    """
    print(f'  h={h:>2}h | RMSE={metrics["rmse"]:6.2f} | '
          f'StormRMSE={metrics["storm_rmse"]:6.2f} | '
          f'MASE={metrics["mase"]:5.3f}')

def run_persistence_horizon(seg_name, seg_mask, h):
    """
    Evaluate persistence at horizon h for one segment.
    Logs a nested MLflow child run and prints progress.
    Returns metrics dict.
    """
    metrics = evaluate_persistence(seg_mask, h)
    log_metrics_run(
        run_name = f'naive_persistence_{seg_name}_h{h}',
        tags     = {'model_type': 'baseline', 'evaluation_set': seg_name},
        params   = {'horizon': h, 'storm_thr': STORM_THR},
        metrics  = metrics,
        nested   = True,
    )
    print_horizon_metrics(h, metrics)
    return metrics

def run_persistence_segment(seg_name, seg_mask):
    """
    Evaluate persistence for all horizons in one segment.
    Opens a parent MLflow run and delegates to run_persistence_horizon().
    Returns dict {horizon: metrics}.
    """
    print(f'\n── Segment: {seg_name} ──────────────────────────────────────')
    with mlflow.start_run(run_name=f'naive_persistence_{seg_name}'):
        mlflow.set_tag('model_type',     'baseline')
        mlflow.set_tag('evaluation_set', seg_name)
        return {h: run_persistence_horizon(seg_name, seg_mask, h) for h in K_HORIZONS}


# ── Run ───────────────────────────────────────────────────────────────────
persistence_model   = NaivePersistence(dst_col='dst')
persistence_metrics = {
    seg_name: run_persistence_segment(seg_name, seg_mask)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(persistence_metrics, 'models/metrics_naive_persistence.pkl')
print('\nSaved: models/metrics_naive_persistence.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE=  3.55 | StormRMSE=  9.19 | MASE=0.754
  h= 3h | RMSE=  7.37 | StormRMSE= 22.34 | MASE=1.587
  h= 7h | RMSE= 10.90 | StormRMSE= 38.85 | MASE=2.300
  h=12h | RMSE= 13.27 | StormRMSE= 50.46 | MASE=2.774
  h=21h | RMSE= 15.44 | StormRMSE= 60.25 | MASE=3.240

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 10.04 | StormRMSE= 24.41 | MASE=1.626
  h= 3h | RMSE= 22.79 | StormRMSE= 58.54 | MASE=3.372
  h= 7h | RMSE= 38.76 | StormRMSE=102.29 | MASE=5.378
  h=12h | RMSE= 47.71 | StormRMSE=126.01 | MASE=6.792
  h=21h | RMSE= 54.42 | StormRMSE=142.70 | MASE=7.997

Saved: models/metrics_naive_persistence.pkl


## Stage 2 — Direct Autoregressive Baseline

$$\hat{D}_{st}(t+h) = \alpha_h \cdot D_{st}(t) + \beta_h$$

The direct autoregressive baseline fits one OLS model per forecast horizon on Train_1. Unlike a classical AR(1) model which propagates one step at a time, this is a direct forecasting model — it predicts $D_{st}(t+h)$ in a single step without iterating through intermediate states. This distinction matters for multi-step horizons where error accumulation in recursive models inflates uncertainty.

**Why this baseline matters:** Persistence assumes the system is static — $\alpha_h = 1$, $\beta_h = 0$. The direct AR baseline relaxes this by learning the actual linear decay rate from data. The fitted $\alpha_h$ coefficient approximates the fraction of the ring current disturbance that persists after $h$ hours, consistent with the Burton et al. (1975) [BUR75] exponential decay model with relaxation time $\tau \approx 7$–8h. If $\alpha_h \approx e^{-h/\tau}$, the model has recovered the physical decay constant from the data without any domain knowledge.

**Why only $D_{st}(t)$ as predictor:** Adding more lags would make this a competitive ML model rather than a baseline. The intentional minimalism ensures that any improvement of XGBoost over the direct AR baseline is attributable to nonlinear solar wind forcing and feature interactions beyond the linear memory of $D_{st}(t)$ alone.

**How it works:** For each horizon $h \in \{1, 3, 7, 12, 21\}$, a separate `DirectARBaseline` instance is fitted on `feat.loc[train1_mask, ['dst']]` with target `dst_target_{h}h`. The input is the raw unscaled `dst` column — `StandardScaler` inside the Pipeline operates only on the single `dst` feature. Predictions are returned in nT. The fitted $\alpha_h$ and $\beta_h$ coefficients are logged to MLflow as params for each child run, allowing direct inspection of the ring current decay structure across horizons.

**Pipeline:**
- `evaluate_ar(seg_mask, h, pipe)` — extracts unscaled `feat.loc[seg_mask, ['dst']]` and `y_true`, calls `pipe.predict()` which applies the fitted scaler and linear model, then passes predictions to `compute_metrics()` with persistence as the DM baseline.
- `log_metrics_run(run_name, tags, params, metrics, nested)` — reused from Stage 1. Logs alpha and beta as params in addition to metrics.
- `print_horizon_metrics(h, metrics)` — reused from Stage 1.
- `run_ar_horizon(seg_name, seg_mask, h, pipe)` — fits `Pipeline([StandardScaler, DirectARBaseline])` on Train_1, evaluates on the segment, delegates logging to `log_metrics_run()` as a nested child run, and calls `print_horizon_metrics()`.
- `run_ar_segment(seg_name, seg_mask)` — opens a parent MLflow run for one segment and delegates to `run_ar_horizon()` for each horizon in `K_HORIZONS`.

**Evaluation design:** Fitted on Train_1 only (121,185 rows). Evaluated on `val_main` and `val_storm` separately with the same segment design as Stage 1. The DM test compares the direct AR model against naive persistence — a significant negative DM statistic indicates that linear Dst memory provides additional predictive value beyond a static forecast.

In [6]:
# ── Cell 5: Direct AR Baseline ────────────────────────────────────────────

def fit_ar_pipeline(h):
    """
    Fit DirectARBaseline on Train_1 for horizon h.
    No scaling — OLS on a single feature in nT units preserves
    physical interpretability of alpha as ring current decay fraction [BUR75].
    Returns fitted DirectARBaseline instance.
    """
    model = DirectARBaseline(dst_col='dst')
    model.fit(
        feat.loc[train1_mask, ['dst']],
        feat.loc[train1_mask, f'dst_target_{h}h'],
    )
    return model

def evaluate_ar(seg_mask, h, model):
    """
    Evaluate fitted DirectARBaseline on one segment at horizon h.
    Uses persistence as DM baseline. Returns metrics dict.
    """
    y_true    = feat.loc[seg_mask, f'dst_target_{h}h'].copy()
    y_pred    = model.predict(feat.loc[seg_mask, ['dst']])
    y_persist = feat.loc[seg_mask, 'dst'].values
    return compute_metrics(
        y_true    = y_true,
        y_pred    = y_pred,
        y_train   = y_train,
        y_persist = y_persist,
        storm_thr = STORM_THR,
        horizon   = h,
    )

def print_ar_horizon_metrics(h, alpha, beta, metrics):
    """
    Print alpha, beta and key metrics for one horizon.
    Alpha approximates ring current decay fraction at this horizon [BUR75].
    Delegates to print_horizon_metrics() for metric formatting.
    """
    print(f'  alpha={alpha:.3f} | beta={beta:.2f}', end=' | ')
    print_horizon_metrics(h, metrics)

def run_ar_horizon(seg_name, seg_mask, h, model):
    """
    Evaluate DirectARBaseline at horizon h for one segment.
    Logs nested MLflow child run with alpha, beta and metrics.
    Reuses log_metrics_run() from Stage 1.
    Returns metrics dict.
    """
    alpha   = model.alpha_
    beta    = model.beta_
    metrics = evaluate_ar(seg_mask, h, model)

    log_metrics_run(
        run_name = f'direct_ar_{seg_name}_h{h}',
        tags     = {'model_type': 'direct_ar', 'evaluation_set': seg_name},
        params   = {'horizon': h, 'storm_thr': STORM_THR,
                    'alpha': round(alpha, 4), 'beta': round(beta, 4)},
        metrics  = metrics,
        nested   = True,
    )
    print_ar_horizon_metrics(h, alpha, beta, metrics)
    return metrics

def run_ar_segment(seg_name, seg_mask, ar_models):
    """
    Evaluate AR baseline for all horizons in one segment.
    Opens parent MLflow run and delegates to run_ar_horizon().
    Returns dict {horizon: metrics}.
    """
    print(f'\n── Segment: {seg_name} ──────────────────────────────────────')
    with mlflow.start_run(run_name=f'direct_ar_{seg_name}'):
        mlflow.set_tag('model_type',     'direct_ar')
        mlflow.set_tag('evaluation_set', seg_name)
        return {h: run_ar_horizon(seg_name, seg_mask, h, ar_models[h])
                for h in K_HORIZONS}


# ── Fit one model per horizon (Train_1 only) ─────────────────────────────
ar_models = {h: fit_ar_pipeline(h) for h in K_HORIZONS}
ar_coefs  = {h: {'alpha': ar_models[h].alpha_,
                  'beta' : ar_models[h].beta_}
             for h in K_HORIZONS}

# ── Evaluate on both validation segments ─────────────────────────────────
ar_metrics = {
    seg_name: run_ar_segment(seg_name, seg_mask, ar_models)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(ar_metrics, 'models/metrics_direct_ar.pkl')
print('\nSaved: models/metrics_direct_ar.pkl')


── Segment: val_main ──────────────────────────────────────
  alpha=0.977 | beta=-0.38 |   h= 1h | RMSE=  3.53 | StormRMSE=  9.26 | MASE=0.763
  alpha=0.902 | beta=-1.61 |   h= 3h | RMSE=  7.20 | StormRMSE= 22.49 | MASE=1.576
  alpha=0.774 | beta=-3.73 |   h= 7h | RMSE= 10.34 | StormRMSE= 38.08 | MASE=2.268
  alpha=0.660 | beta=-5.62 |   h=12h | RMSE= 12.25 | StormRMSE= 47.81 | MASE=2.727
  alpha=0.526 | beta=-7.83 |   h=21h | RMSE= 13.83 | StormRMSE= 54.98 | MASE=3.167

── Segment: val_storm ──────────────────────────────────────
  alpha=0.977 | beta=-0.38 |   h= 1h | RMSE= 10.00 | StormRMSE= 24.39 | MASE=1.605
  alpha=0.902 | beta=-1.61 |   h= 3h | RMSE= 22.26 | StormRMSE= 57.44 | MASE=3.157
  alpha=0.774 | beta=-3.73 |   h= 7h | RMSE= 36.15 | StormRMSE= 95.87 | MASE=4.747
  alpha=0.660 | beta=-5.62 |   h=12h | RMSE= 42.75 | StormRMSE=113.62 | MASE=5.761
  alpha=0.526 | beta=-7.83 |   h=21h | RMSE= 46.80 | StormRMSE=124.34 | MASE=6.555

Saved: models/metrics_direct_ar.pkl


## Stage 3 — XGBoost OMNI (MODEL_A)

$$\text{MODEL\_A}: \hat{D}_{st}(t+h) = f_{\text{XGB}}(\mathbf{x}_{\text{OMNI}}(t))$$

XGBoost trained on OMNI solar wind features only (MODEL_A_OMNI, 19 features). No hyperparameter tuning at this stage — default parameters are used intentionally to establish a clean nonlinear solar wind predictability baseline before neutron features are introduced. Tuning is deferred to Stage 6 after horizon selection to avoid optimising on the wrong target.

**Why XGBoost before tuning:** The goal of Stage 3 is not to maximise performance but to answer a specific question — how much predictability does nonlinear solar wind forcing add above the linear Dst memory established in Stage 2? A default XGBoost model is sufficient to quantify this gap. Premature tuning would conflate the contribution of the model architecture with the contribution of the feature set.

**Why storm sample weighting:** Storm hours ($D_{st} < -50$ nT) represent only 4.71% of training data. An unweighted XGBoost fit suppresses the storm signal and learns primarily from quiet-time dynamics. `XGBoostDst.fit()` applies inverse-frequency weights ($w = 1/f_{\text{storm}} \approx 21.2\times$) to upweight storm hours, consistent with the weighting strategy used in feature selection [KIS25].

**How it works:** For each horizon $h \in \{1, 3, 7, 12, 21\}$, a separate `Pipeline([StandardScaler, XGBoostDst])` is fitted on `feat.loc[train1_mask, MODEL_A_OMNI]`. The `StandardScaler` is fitted inside the Pipeline on Train_1 only — no leakage from validation or test segments. `XGBoostDst` receives scaled features and applies storm sample weighting internally during `fit()`. Predictions are returned in nT.

**Pipeline:**
- `fit_xgb_pipeline(feature_cols, h)` — fits `Pipeline([StandardScaler, XGBoostDst])` on Train_1 for one horizon. Storm weights are applied inside `XGBoostDst.fit()`. Returns fitted pipeline.
- `evaluate_xgb(seg_mask, h, pipe, feature_cols)` — evaluates fitted pipeline on one segment at horizon h. Uses persistence as DM baseline. Returns metrics dict from `compute_metrics()`.
- `log_xgb_model(pipe, h)` — logs fitted pipeline as MLflow artifact using skops serialisation. Trusted types are declared explicitly for the custom `XGBoostDst` class.
- `run_xgb_horizon(seg_name, seg_mask, h, pipe, feature_set_name, feature_cols)` — evaluates pipeline at one horizon, logs nested MLflow child run with params, metrics and pipeline artifact. Reuses `log_metrics_run()` and `print_horizon_metrics()` from Stage 1.
- `run_xgb_segment(seg_name, seg_mask, xgb_pipes, feature_set_name, feature_cols)` — opens parent MLflow run for one segment and delegates to `run_xgb_horizon()` for each horizon in `K_HORIZONS`.

**Evaluation design:** Fitted on Train_1 only (121,185 rows). Evaluated on `val_main` and `val_storm` separately. The DM test compares MODEL_A against naive persistence — a significant negative DM statistic indicates that nonlinear solar wind forcing provides predictive value beyond a static forecast. Results establish the OMNI-only performance ceiling that MODEL_C must surpass to support H_gain.

In [7]:
# ── Cell 6: XGBoost MODEL_A (OMNI only) ──────────────────────────────────

SKOPS_TRUSTED_TYPES = [
    'src.estimators.xgboost_dst.XGBoostDst',
    'xgboost.core.Booster',
    'xgboost.sklearn.XGBRegressor',
]

def fit_xgb_pipeline(feature_cols, h):
    """
    Fit Pipeline([XGBoostDst]) on Train_1 for horizon h.
    No scaling — tree-based models are invariant to monotone feature
    transformations. Raw physical units preserved for SHAP interpretability.
    Storm sample weights applied internally in XGBoostDst.fit().
    Returns fitted Pipeline.
    """
    pipe = Pipeline([
        ('model', XGBoostDst()),
    ])
    pipe.fit(
        feat.loc[train1_mask, feature_cols],
        feat.loc[train1_mask, f'dst_target_{h}h'],
    )
    return pipe

def evaluate_xgb(seg_mask, h, pipe, feature_cols):
    """
    Evaluate fitted Pipeline on one segment at horizon h.
    Uses persistence as DM baseline. Returns metrics dict.
    """
    y_true    = feat.loc[seg_mask, f'dst_target_{h}h'].copy()
    y_pred    = pipe.predict(feat.loc[seg_mask, feature_cols])
    y_persist = feat.loc[seg_mask, 'dst'].values
    return compute_metrics(
        y_true    = y_true,
        y_pred    = y_pred,
        y_train   = y_train,
        y_persist = y_persist,
        storm_thr = STORM_THR,
        horizon   = h,
    )

def log_xgb_model(pipe, h):
    """
    Log fitted Pipeline as MLflow artifact using skops serialisation.
    Trusted types declared explicitly for custom XGBoostDst class.
    """
    mlflow.sklearn.log_model(
        pipe,
        name=f'pipeline_h{h}',
        skops_trusted_types=SKOPS_TRUSTED_TYPES,
    )

def run_xgb_horizon(seg_name, seg_mask, h, pipe, feature_set_name, feature_cols):
    """
    Evaluate Pipeline at horizon h for one segment.
    Logs nested MLflow child run with params, metrics and pipeline artifact.
    Reuses log_metrics_run() and print_horizon_metrics() from Stage 1.
    Returns metrics dict.
    """
    metrics = evaluate_xgb(seg_mask, h, pipe, feature_cols)

    log_metrics_run(
        run_name = f'xgb_{feature_set_name.lower()}_{seg_name}_h{h}',
        tags     = {'model_type'    : 'xgboost',
                    'feature_set'   : feature_set_name,
                    'evaluation_set': seg_name},
        params   = {'horizon'    : h,
                    'feature_set': feature_set_name,
                    'n_features' : len(feature_cols),
                    'storm_thr'  : STORM_THR},
        metrics  = metrics,
        nested   = True,
    )
    log_xgb_model(pipe, h)
    print_horizon_metrics(h, metrics)
    return metrics

def run_xgb_segment(seg_name, seg_mask, xgb_pipes, feature_set_name, feature_cols):
    """
    Evaluate XGBoost Pipeline for all horizons in one segment.
    Opens parent MLflow run and delegates to run_xgb_horizon().
    Returns dict {horizon: metrics}.
    """
    print(f'\n── Segment: {seg_name} ──────────────────────────────────────')
    with mlflow.start_run(run_name=f'xgb_{feature_set_name.lower()}_{seg_name}'):
        mlflow.set_tag('model_type',     'xgboost')
        mlflow.set_tag('feature_set',    feature_set_name)
        mlflow.set_tag('evaluation_set', seg_name)
        return {h: run_xgb_horizon(seg_name, seg_mask, h, xgb_pipes[h],
                                   feature_set_name, feature_cols)
                for h in K_HORIZONS}


# ── Fit one pipeline per horizon (Train_1 only) ───────────────────────────
model_a_pipes = {h: fit_xgb_pipeline(MODEL_A_OMNI, h) for h in K_HORIZONS}

# ── Evaluate on both validation segments ──────────────────────────────────
model_a_metrics = {
    seg_name: run_xgb_segment(seg_name, seg_mask, model_a_pipes,
                               'MODEL_A_OMNI', MODEL_A_OMNI)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_a_metrics, 'models/metrics_xgb_model_a.pkl')
print('\nSaved: models/metrics_xgb_model_a.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.95 | StormRMSE= 18.90 | MASE=2.729
  h= 3h | RMSE= 11.73 | StormRMSE= 19.85 | MASE=2.874
  h= 7h | RMSE= 14.47 | StormRMSE= 29.72 | MASE=3.450
  h=12h | RMSE= 16.39 | StormRMSE= 39.63 | MASE=3.849
  h=21h | RMSE= 19.29 | StormRMSE= 49.37 | MASE=4.682

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 29.52 | StormRMSE= 75.99 | MASE=4.570
  h= 3h | RMSE= 33.73 | StormRMSE= 86.70 | MASE=5.026
  h= 7h | RMSE= 40.76 | StormRMSE=107.04 | MASE=5.581
  h=12h | RMSE= 45.31 | StormRMSE=119.15 | MASE=6.368
  h=21h | RMSE= 50.86 | StormRMSE=132.81 | MASE=7.294

Saved: models/metrics_xgb_model_a.pkl


## Stage 4 — XGBoost OMNI + $\delta n$ (MODEL_C)

$$\text{MODEL\_C}: \hat{D}_{st}(t+h) = f_{\text{XGB}}(\mathbf{x}_{\text{OMNI}}(t), \delta n(t))$$

XGBoost trained on OMNI solar wind features plus the differential neutron flux $\delta n(t)$ (MODEL_C_OMNI_DNEUTRON, 20 features). The architecture is identical to MODEL_A — same Pipeline structure, same default hyperparameters, same storm sample weighting, same Train_1 training set. Only the feature set changes. This design ensures that any difference in metrics between MODEL_A and MODEL_C is attributable solely to the information content of $\delta n(t)$ and not to architectural differences.

**Why $\delta n(t)$ and not raw neutron counts:** The differential neutron flux $\delta n(t) = (N(t) - N(t-7)) / N(t-7)$ captures the rate of change of the cosmic ray flux over a 7-hour window — physically motivated by the Forbush Decrease lead time of 7–21h established in [KIS25]. Raw neutron counts carry a strong solar cycle trend that is already partially encoded in `f107` and `ssn`. The differential formulation isolates the event-driven transient component relevant to geomagnetic storm prediction.

**Why $\delta n$ is not in SELECTED_FEATURES:** Feature selection via storm-weighted LASSO and ExtraTrees did not include $\delta n$ in the strict intersection (votes=2). LASSO retains it at h=7h (coefficient 0.174) but ExtraTrees importance falls below the median threshold at all horizons (0.003–0.010 vs threshold 0.017–0.027). This reflects the event-driven nature of the Forbush Decrease signal — split-gain importance is computed over the full training set where quiet periods dominate (95.3%), systematically underestimating features whose signal is concentrated in 0.619% of records. Force-inclusion in MODEL_C is warranted on physical grounds [KIS25] and constitutes the primary H_gain test.

**How it works:** For each horizon $h \in \{1, 3, 7, 12, 21\}$, a separate `Pipeline([StandardScaler, XGBoostDst])` is fitted on `feat.loc[train1_mask, MODEL_C_OMNI_DNEUTRON]`. All functions from Stage 3 are reused directly — `fit_xgb_pipeline`, `evaluate_xgb`, `log_xgb_model`, `run_xgb_horizon` and `run_xgb_segment` — with `feature_cols=MODEL_C_OMNI_DNEUTRON` and `feature_set_name='MODEL_C_OMNI_DNEUTRON'` as the only differences.

**Pipeline:** Identical to Stage 3 — all five functions are reused without modification. See Stage 3 for full pipeline description.

**Evaluation design:** Fitted on Train_1 only (121,185 rows). Evaluated on `val_main` and `val_storm` separately. The primary comparison is MODEL_C vs MODEL_A — $\Delta R^2 = R^2(\text{MODEL\_C}) - R^2(\text{MODEL\_A})$ at each horizon quantifies the information gain from $\delta n(t)$. Positive $\Delta R^2$ at $h \geq 7$h supports H_gain. The DM test provides statistical significance of the improvement.

In [8]:
# ── Cell 7: XGBoost MODEL_C (OMNI + δn) ──────────────────────────────────
# All functions reused from Stage 3 (Cell 6) — only feature set changes.

# ── Fit one pipeline per horizon (Train_1 only) ───────────────────────────
model_c_pipes = {h: fit_xgb_pipeline(MODEL_C_OMNI_DNEUTRON, h) for h in K_HORIZONS}

# ── Evaluate on both validation segments ──────────────────────────────────
model_c_metrics = {
    seg_name: run_xgb_segment(seg_name, seg_mask, model_c_pipes,
                               'MODEL_C_OMNI_DNEUTRON', MODEL_C_OMNI_DNEUTRON)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_c_metrics, 'models/metrics_xgb_model_c.pkl')
print('\nSaved: models/metrics_xgb_model_c.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.89 | StormRMSE= 18.58 | MASE=2.718
  h= 3h | RMSE= 11.56 | StormRMSE= 19.89 | MASE=2.834
  h= 7h | RMSE= 13.83 | StormRMSE= 28.79 | MASE=3.349
  h=12h | RMSE= 16.02 | StormRMSE= 38.91 | MASE=3.785
  h=21h | RMSE= 19.40 | StormRMSE= 48.84 | MASE=4.721

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 29.68 | StormRMSE= 76.98 | MASE=4.512
  h= 3h | RMSE= 32.76 | StormRMSE= 84.16 | MASE=4.928
  h= 7h | RMSE= 40.77 | StormRMSE=106.54 | MASE=5.681
  h=12h | RMSE= 46.06 | StormRMSE=121.54 | MASE=6.378
  h=21h | RMSE= 51.19 | StormRMSE=134.37 | MASE=7.308

Saved: models/metrics_xgb_model_c.pkl


## Stage 5 — Comparison Table & $\Delta R^2$ Analysis

The comparison table consolidates results from Stages 1–4 across all horizons and both validation segments. It serves two purposes: first, to establish whether any model provides predictive value above the baselines (MASE < 1, positive R²); second, to quantify the information gain from $\delta n(t)$ via $\Delta R^2$.

**Primary comparison — H_gain:**

$$\Delta R^2(h) = R^2(\text{MODEL\_C}) - R^2(\text{MODEL\_A})$$

Positive $\Delta R^2$ at $h \geq 7$h indicates that $\delta n(t)$ adds predictive information above OMNI solar wind parameters alone, supporting H_gain. The sign and magnitude of $\Delta R^2$ across horizons reveals at which lead times the Forbush Decrease signal is most informative relative to the geomagnetic storm onset.

**Predictability hierarchy:** The four-model comparison establishes a clean hierarchy of information sources:
- Persistence → linear Dst memory (AR) → nonlinear solar wind forcing (MODEL_A) → neutron flux contribution (MODEL_C)

Each step isolates a distinct source of predictability. If MODEL_A ≈ AR, nonlinear solar wind forcing adds little. If MODEL_C ≈ MODEL_A, $\delta n(t)$ adds little. The incremental gains at each step motivate the subsequent modelling decisions.

**Horizon selection $h^*$:** The comparison table is the primary input to Stage 6. The operating horizon $h^*$ is selected based on three criteria evaluated jointly: positive $\Delta R^2$ (H_gain signal present), physically motivated lead time consistent with [KIS25], and sufficient absolute predictability (positive R² for MODEL_A). Horizons where both models produce negative R² indicate that the forecasting problem exceeds the capacity of default XGBoost on Train_1 alone and require tuning before conclusions can be drawn.

**How it works:** `get_r2(seg_mask, feature_cols, pipe, h)` computes R² by extracting finite `y_true` values, predicting with the fitted pipeline and calling `r2_score()`. The $\Delta R^2$ table iterates over `K_HORIZONS`, calls `get_r2()` for MODEL_A and MODEL_C pipelines, and prints the signed difference. Both `val_main` and `val_storm` are reported to assess whether the neutron gain generalises to extreme events.

In [9]:
def get_model_metrics(metrics_dict, seg_name, h):
    """Extract RMSE and Storm RMSE for one model/segment/horizon."""
    m = metrics_dict[seg_name][h]
    return round(m['rmse'], 2), round(m['storm_rmse'], 2)

def build_comparison_row(seg_name, h):
    """
    Build one comparison row for horizon h across all four models.
    Returns dict ready for DataFrame construction.
    """
    p_rmse,  p_srmse  = get_model_metrics(persistence_metrics, seg_name, h)
    ar_rmse, ar_srmse = get_model_metrics(ar_metrics,          seg_name, h)
    a_rmse,  a_srmse  = get_model_metrics(model_a_metrics,     seg_name, h)
    c_rmse,  c_srmse  = get_model_metrics(model_c_metrics,     seg_name, h)

    return {
        'h'                    : h,
        'Persistence RMSE'     : p_rmse,
        'Persistence StormRMSE': p_srmse,
        'AR RMSE'              : ar_rmse,
        'AR StormRMSE'         : ar_srmse,
        'XGB-A RMSE'           : a_rmse,
        'XGB-A StormRMSE'      : a_srmse,
        'XGB-C RMSE'           : c_rmse,
        'XGB-C StormRMSE'      : c_srmse,
    }

def print_comparison_table(seg_name):
    """Build and print comparison table for one segment."""
    rows = [build_comparison_row(seg_name, h) for h in K_HORIZONS]
    df   = pd.DataFrame(rows).set_index('h')
    print(f'\n── {seg_name} ──────────────────────────────────────')
    print(df.to_string())

for seg_name in EVAL_SEGMENTS:
    print_comparison_table(seg_name)


── val_main ──────────────────────────────────────
    Persistence RMSE  Persistence StormRMSE  AR RMSE  AR StormRMSE  XGB-A RMSE  XGB-A StormRMSE  XGB-C RMSE  XGB-C StormRMSE
h                                                                                                                           
1               3.55                   9.19     3.53          9.26       10.95            18.90       10.89            18.58
3               7.37                  22.34     7.20         22.49       11.73            19.85       11.56            19.89
7              10.90                  38.85    10.34         38.08       14.47            29.72       13.83            28.79
12             13.27                  50.46    12.25         47.81       16.39            39.63       16.02            38.91
21             15.44                  60.25    13.83         54.98       19.29            49.37       19.40            48.84

── val_storm ──────────────────────────────────────
    Persistence RMSE

In [10]:
# ── Cell 9: ΔR² table — H_gain analysis ──────────────────────────────────

def compute_delta_r2(seg_name, h):
    """
    Extract R²(MODEL_A), R²(MODEL_C) and ΔR² from already computed metrics.
    No recomputation needed — r2 is included in compute_metrics() output.
    Returns tuple (r2_a, r2_c, delta).
    """
    r2_a = model_a_metrics[seg_name][h]['r2']
    r2_c = model_c_metrics[seg_name][h]['r2']
    return r2_a, r2_c, r2_c - r2_a

def print_delta_r2_table(seg_name):
    """
    Print ΔR² table for one segment across all horizons.
    Positive ΔR² at h≥7h supports H_gain hypothesis.
    """
    print(f'\nΔR² = R²(OMNI+δn) - R²(OMNI)  [{seg_name}]')
    print('=' * 45)
    print(f'{"h":>4} | {"R²(A)":>8} | {"R²(C)":>8} | {"ΔR²":>8}')
    print('-' * 45)
    for h in K_HORIZONS:
        r2_a, r2_c, delta = compute_delta_r2(seg_name, h)
        print(f'{h:>4}h | {r2_a:>8.4f} | {r2_c:>8.4f} | {delta:>+8.4f}')

for seg_name in EVAL_SEGMENTS:
    print_delta_r2_table(seg_name)

print('\nPositive ΔR² at h≥7h supports H_gain.')


ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_main]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.5058 |   0.5116 |  +0.0058
   3h |   0.4332 |   0.4493 |  +0.0161
   7h |   0.1376 |   0.2123 |  +0.0747
  12h |  -0.1066 |  -0.0574 |  +0.0492
  21h |  -0.5323 |  -0.5500 |  -0.0178

ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_storm]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.6453 |   0.6413 |  -0.0040
   3h |   0.5369 |   0.5632 |  +0.0263
   7h |   0.3236 |   0.3233 |  -0.0003
  12h |   0.1645 |   0.1366 |  -0.0278
  21h |  -0.0525 |  -0.0662 |  -0.0137

Positive ΔR² at h≥7h supports H_gain.


> **Observations — Direct AR Baseline (Stage 2):**
> - **Alpha coefficients are physically interpretable:** $\alpha_h$ decreases monotonically from 0.977 at h=1h to 0.526 at h=21h, consistent with exponential ring current decay. The fitted values follow $\alpha_h \approx e^{-h/\tau}$ with $\tau \approx 38$h — somewhat longer than the Burton et al. (1975) [BUR75] theoretical value of $\tau \approx 7$–8h, which reflects that the empirical decay rate over the full training set integrates both active and quiet periods. During storms the decay is faster; during quiet times Dst is nearly stationary, pulling $\alpha$ upward.
> - **AR marginally outperforms persistence** at all horizons on val_main (e.g. h=7h: AR Storm RMSE=38.08 vs Persistence=38.85). The improvement is small, confirming that linear Dst memory alone adds limited predictive value beyond persistence.
> - **val_storm:** AR substantially outperforms persistence at longer horizons (h=21h: AR Storm RMSE=124.34 vs Persistence=142.70), suggesting that during extreme storms the linear decay structure is more informative.

> **Observations — XGBoost Horizon Screening (Stages 3 & 4):**
> - **XGBoost underperforms persistence and AR on val_main** at all horizons with default hyperparameters and Train_1 only. At h=7h Storm RMSE=29.72 (MODEL_A) vs 38.08 (AR) — XGBoost does outperform AR on storm hours, which is the primary metric.
> - **H_gain signal is strongest at h=7h on val_main:** $\Delta R^2 = +0.0747$ — the largest positive increment across all horizons. MODEL_C (OMNI+δn) reduces Storm RMSE from 29.72 to 28.79 nT at h=7h.
> - **val_storm H_gain is mixed:** positive $\Delta R^2$ only at h=3h (+0.0263). At h=7h $\Delta R^2 \approx 0$ — the neutron signal does not generalise to the Halloween 2003 superstorm with default hyperparameters. This does not reject H_gain — tuning on Train_1+Train_2 (Stage 6) may recover the signal.
> - **h\*=7h confirmed** as the primary operating horizon: largest $\Delta R^2$ on val_main, physically motivated lead time [KIS25], and positive absolute R² for MODEL_C.

## Stage 6 — Ablation Study

The ablation study decomposes the neutron flux contribution by comparing four feature sets with identical XGBoost architecture and training protocol. The goal is to determine which neutron representation carries the most predictive value and whether the physical transformation $\delta n(t)$ is necessary or whether raw neutron counts are sufficient.

**Relationship to feature selection:** The feature selection stage (LASSO + ExtraTrees) operated on the full feature set and identified which individual features carry predictive signal — it did not compare neutron representations as competing feature sets. At h=7h, `neutron_counts`, `neutron_counts_lag3` and `neutron_counts_lag7` survive the strict intersection (votes=2), while `d_neutron` survives only LASSO (coefficient 0.174, α=0.311) but falls below the ExtraTrees median importance threshold (0.0096 vs threshold 0.0235). The ablation study here addresses a different question: given that some neutron features survive selection, which representation of the neutron signal is most informative as a group? This is a model-level comparison, not a feature-level one.

**Experimental matrix:**

| Model | Features | Scientific question |
|---|---|---|
| MODEL_A | OMNI only | Reference — no neutron information |
| MODEL_B | OMNI + raw neutron counts | Does the absolute flux level add signal? |
| MODEL_C | OMNI + $\delta n(t)$ | Does the rate-of-change add signal? |
| MODEL_D | OMNI + $\delta n(t)$ + neutron lags | Do historical neutron lags add signal beyond $\delta n$? |

**Why this ordering matters:** MODEL_B vs MODEL_A isolates the raw flux level. MODEL_C vs MODEL_A isolates the physically motivated differential flux. MODEL_D vs MODEL_C isolates the contribution of neutron lag history. If MODEL_C > MODEL_B, the physical transformation $\delta n$ is justified. If MODEL_D ≈ MODEL_C, the lag history adds no independent information beyond $\delta n(t)$.

**How it works:** All four models are fitted on Train_1 only with identical default hyperparameters and storm sample weighting. Evaluation is performed on val_main and val_storm separately. All functions from Stages 3–4 are reused — `fit_xgb_pipeline`, `evaluate_xgb`, `log_xgb_model`, `run_xgb_horizon` and `run_xgb_segment` — with the feature set as the only variable.

> **Note:** MODEL_A and MODEL_C metrics are already computed in Stages 3–4 and are loaded directly from `models/metrics_xgb_model_a.pkl` and `models/metrics_xgb_model_c.pkl`. Only MODEL_B and MODEL_D require new training runs.

In [11]:
# ── Cell: Ablation Study — MODEL_B and MODEL_D ───────────────────────────
# MODEL_A and MODEL_C metrics already computed in Stages 3-4.
# Only MODEL_B (OMNI + raw neutron) and MODEL_D (OMNI + δn + lags) are new.
# All functions reused from Stage 3 (Cell 6).

# ── Fit MODEL_B (OMNI + raw neutron counts) ──────────────────────────────
model_b_pipes = {h: fit_xgb_pipeline(MODEL_B_OMNI_RAW, h) for h in K_HORIZONS}

model_b_metrics = {
    seg_name: run_xgb_segment(seg_name, seg_mask, model_b_pipes,
                               'MODEL_B_OMNI_RAW', MODEL_B_OMNI_RAW)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_b_metrics, 'models/metrics_xgb_model_b.pkl')
print('Saved: models/metrics_xgb_model_b.pkl')

# ── Fit MODEL_D (OMNI + δn + neutron lags) ───────────────────────────────
model_d_pipes = {h: fit_xgb_pipeline(MODEL_D_FULL_NEUTRON, h) for h in K_HORIZONS}

model_d_metrics = {
    seg_name: run_xgb_segment(seg_name, seg_mask, model_d_pipes,
                               'MODEL_D_FULL_NEUTRON', MODEL_D_FULL_NEUTRON)
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_d_metrics, 'models/metrics_xgb_model_d.pkl')
print('Saved: models/metrics_xgb_model_d.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.49 | StormRMSE= 18.98 | MASE=2.871
  h= 3h | RMSE= 12.43 | StormRMSE= 19.49 | MASE=3.063
  h= 7h | RMSE= 14.59 | StormRMSE= 29.85 | MASE=3.500
  h=12h | RMSE= 16.41 | StormRMSE= 40.36 | MASE=3.785
  h=21h | RMSE= 17.54 | StormRMSE= 51.61 | MASE=4.172

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 29.47 | StormRMSE= 75.97 | MASE=4.611
  h= 3h | RMSE= 33.35 | StormRMSE= 85.99 | MASE=5.005
  h= 7h | RMSE= 39.80 | StormRMSE=101.96 | MASE=6.095
  h=12h | RMSE= 46.08 | StormRMSE=118.88 | MASE=7.126
  h=21h | RMSE= 51.30 | StormRMSE=130.04 | MASE=8.117
Saved: models/metrics_xgb_model_b.pkl

── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.37 | StormRMSE= 18.49 | MASE=2.850
  h= 3h | RMSE= 11.91 | StormRMSE= 19.55 | MASE=2.925
  h= 7h | RMSE= 14.66 | StormRMSE= 30.33 | MASE=3.565
  h=12h | RMSE= 16.51 | StormRMSE= 40.89 | MASE=3.851
  h=21h | RMSE= 17.84 

> **Observations — Ablation Study (h=7h, val_main):**
> - **MODEL_C (OMNI + δn) is the best neutron representation** — lowest RMSE (13.83 nT) and Storm RMSE (28.79 nT) at h=7h, outperforming all other variants including MODEL_A.
> - **MODEL_B (OMNI + raw counts) performs worse than MODEL_A** — Storm RMSE=29.85 vs 29.72 nT. Raw neutron counts carry no additional signal above OMNI alone, confirming that the physical transformation $\delta n(t)$ is necessary.
> - **MODEL_D (OMNI + δn + lags) underperforms MODEL_C** — Storm RMSE=30.33 vs 28.79 nT. Adding neutron lag history introduces noise rather than signal with default hyperparameters, suggesting that $\delta n(t)$ alone captures the relevant Forbush Decrease precursor information.
> - **Conclusion:** The differential neutron flux $\delta n(t)$ is the optimal neutron representation. MODEL_C is confirmed as the primary H_gain model for Part II.
>
> - **Physical interpretation:** MODEL_B (OMNI + raw counts, Storm RMSE=29.85 nT at h=7h) performs virtually identically to MODEL_A (OMNI only, Storm RMSE=29.72 nT) — a difference of 0.13 nT, well within the expected variance of a single training run. Raw neutron counts carry no measurable information above OMNI solar wind parameters. MODEL_C (OMNI + $\delta n$, Storm RMSE=28.79 nT) is the only neutron representation that adds a consistent signal — an improvement of 0.93 nT over MODEL_A. This is consistent with the interpretation that the Forbush Decrease precursor signal is encoded in the *rate of change* of the cosmic ray flux, not in the absolute flux level. The same plasma structure that causes a rapid cosmic ray depression will subsequently compress Earth's magnetosphere — $\delta n(t)$ captures this transient, whereas raw counts conflate it with the slowly varying solar cycle background already encoded in `f107` and `ssn`. This is consistent with [KIS25] where the neutron monitor correlation with $D_{st}$ strengthens specifically during Forbush Decrease periods.

## Stage 7 — Horizon Selection

The operating horizon $h^*$ is selected based on three criteria evaluated jointly across Stages 1–6: predictability (positive R²), neutron gain ($\Delta R^2 > 0$), and physical motivation.

**Summary of evidence at h=7h:**

| Criterion | Value | Assessment |
|---|---|---|
| Persistence Storm RMSE | 38.85 nT | Reference lower bound |
| Direct AR Storm RMSE | 38.08 nT | Marginal improvement over persistence |
| XGBoost MODEL_A Storm RMSE | 29.72 nT | Substantial improvement over AR |
| XGBoost MODEL_C Storm RMSE | 28.79 nT | Best neutron representation |
| $\Delta R^2$ (MODEL_C vs MODEL_A) | +0.0747 | Largest positive increment across all horizons |
| $\alpha_{h=7}$ (Direct AR) | 0.774 | 77% ring current memory at 7h |

**Why h=7h and not h=3h or h=12h:**

At h=3h the $\Delta R^2 = +0.0161$ — the neutron gain is present but small. The forecast horizon is too short for the Forbush Decrease precursor signal to provide meaningful lead time over real-time solar wind observations. At h=12h $\Delta R^2 = +0.0492$ but R²(MODEL_A) = −0.1066, meaning the OMNI-only model already underperforms persistence at this horizon with default hyperparameters. The neutron gain is positive but the absolute predictability is insufficient.

At h=7h both conditions are satisfied simultaneously: the neutron gain is the largest across all horizons ($\Delta R^2 = +0.0747$) and the absolute predictability remains positive (R²(MODEL_C) = +0.2123). The 7-hour lead time is also physically motivated — [KIS25] establishes that the neutron monitor correlation with $D_{st}$ is maximal at a delay of 7–21h, and the Burton et al. (1975) [BUR75] ring current decay timescale $\tau \approx 7$–8h defines the natural timescale of geomagnetic storm evolution.

**Selected horizon:** $h^* = 7$h. MODEL_C (OMNI + $\delta n$) is the feature set carried forward to Part II.